# Taller integrador · ¿De dónde sale el 27.6 %?

**Estadística Descriptiva e Inferencial** · Taller · 120 minutos

Reconstruir la cifra oficial de pobreza del Perú desde el microdato, y después
averiguar qué tan segura es.

---

## El titular

> «La pobreza monetaria en el Perú afectó al **27.6 %** de la población en 2024,
> frente al 29.0 % del año anterior.»

Ese número sale de la **ENAHO**, una encuesta a unos 34 000 hogares. Hoy vas a hacer
cuatro cosas con él:

1. **Reproducirlo** exactamente, desde el gasto de cada hogar.
2. **Ponerle un intervalo**, y descubrir que es más ancho de lo que parece.
3. **Comparar grupos**: urbano contra rural, y 2024 contra 2023.
4. **Escribir el informe** que un comité podría auditar.

## Lo que este taller integra

| Bloque | Min | Qué usas de las clases anteriores |
|---|---|---|
| 0 · El titular y los datos | 12 | — |
| 1 · **Factores de expansión** | 28 | tema nuevo |
| 2 · Reconstruir la cifra | 20 | Clases 1, 2 y 3 |
| 3 · ¿Qué tan seguro es? | 22 | Clases 3 y 4 |
| 4 · Comparar grupos | 25 | Clases 5 y 6 |
| 5 · El informe | 13 | Clases 4, 5 y 6 |

> **Formato:** este notebook se trabaja **en vivo**. Yo ejecuto y explico; tú sigues y
> completas los huecos marcados con `# ── TU CÓDIGO ──`. Las celdas de verificación te
> avisan si un número no coincide.

---
## Celda 0 · Preparación y carga de datos

Ejecuta esta celda. Intenta descargar el dataset del repo del curso; si falla, te
permite subirlo a mano.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

SEED = 42
rng = np.random.default_rng(SEED)

NAVY, BLUE, MAG, GREEN = "#0A2559", "#1A56E8", "#E6115E", "#12B886"
plt.rcParams.update({
    "figure.figsize": (9, 4.2), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})
pd.set_option("display.width", 140)

# ── URL del dataset ──────────────────────────────────────────────────────
URL = ("https://github.com/josefrodrim/Estad-stica-Descriptiva-E-Inferencial/"
       "blob/main/Taller_7/Data/enaho_taller.csv.gz")

def a_raw(url):
    """GitHub sirve HTML en /blob/. Lo convierte al enlace de descarga directa."""
    if "github.com" in url and "/blob/" in url:
        url = (url.replace("github.com", "raw.githubusercontent.com")
                  .replace("/blob/", "/"))
    return url

def cargar():
    try:
        d = pd.read_csv(a_raw(URL))
        print("Datos cargados desde el repo del curso.")
        return d
    except Exception as e:
        print(f"No se pudo descargar ({type(e).__name__}).")
        print("Sube el archivo enaho_taller.csv.gz con el botón de archivos de Colab,")
        print("o ejecuta: from google.colab import files; files.upload()")
        try:
            return pd.read_csv("enaho_taller.csv.gz")
        except Exception:
            raise SystemExit("Carga el archivo y vuelve a ejecutar esta celda.")

df = cargar()
print(f"\n{len(df):,} hogares · {len(df.columns)} variables · años {sorted(df['anio'].unique())}")

# ── Verificador ──────────────────────────────────────────────────────────
def check(nombre, obtenido, esperado, tol=1e-4):
    if obtenido is None:
        print(f"[ ] {nombre}: todavia no calculaste nada (None)")
        return False
    ok = abs(float(obtenido) - float(esperado)) <= tol
    print(f"{'[OK]' if ok else '[X ]'} {nombre}: obtenido = {float(obtenido):,.4f} | "
          f"esperado = {float(esperado):,.4f}")
    if not ok:
        print("      -> revisa este paso antes de continuar.")
    return ok

def check_bool(nombre, cond, pista=""):
    print(f"{'[OK]' if cond else '[X ]'} {nombre}")
    if not cond and pista:
        print(f"      -> {pista}")
    return bool(cond)

# ── Mediana ponderada: numpy no la trae ──────────────────────────────────
def mediana_ponderada(x, w):
    """Mediana de x ponderada por w, para estimar cuantiles poblacionales."""
    x = np.asarray(x, dtype=float); w = np.asarray(w, dtype=float)
    o = np.argsort(x); x, w = x[o], w[o]
    return float(np.interp(0.5, np.cumsum(w) / w.sum(), x))

### El diccionario de este dataset

Es un extract de la **ENAHO 2023 y 2024**, módulo 34 (Sumarias) unido al módulo 01
(NBI, solo 2024). Un registro por hogar.

| Variable | Qué es |
|---|---|
| `anio` | 2023 o 2024 |
| `conglome` | **conglomerado**: el área geográfica donde se sortearon los hogares |
| `estrato` | estrato geográfico (1–5 urbano, 6–8 rural) |
| `dominio`, `dominio_nom` | 8 dominios: Costa/Sierra/Selva y Lima Metropolitana |
| `area` | urbano o rural |
| `dpto` | departamento |
| `mieperho` | número de miembros del hogar |
| `gashog2d` | **gasto total anual del hogar**, en soles |
| `linea` | **línea de pobreza** mensual per cápita, según dominio |
| `linpe` | línea de pobreza *extrema* (solo alimentaria) |
| `pobreza` | 1 = pobre extremo, 2 = pobre no extremo, 3 = no pobre |
| `pobre` | 1 si el hogar es pobre (`pobreza <= 2`) |
| `factor07` | **factor de expansión**: a cuántos hogares del país representa |
| `w_hog` | = `factor07` (peso de hogares) |
| `w_per` | = `factor07 × mieperho` (peso de **personas**) |
| `nbi1`…`nbi5`, `n_nbi`, `pobre_nbi` | necesidades básicas insatisfechas (solo 2024) |

**Fuente:** INEI, ENAHO, módulos `906-Modulo34`, `966-Modulo34` y `966-Modulo01`.

---
# Bloque 0 · Mira los datos antes de tocarlos  ·  12 min

Regla que arrastramos desde la Clase 1: primero se mira, después se calcula.

In [ ]:
# ── DEMOSTRACIÓN ─────────────────────────────────────────────────────────
d24 = df[df.anio == 2024].copy()
d23 = df[df.anio == 2023].copy()

print(f"2024: {len(d24):,} hogares    2023: {len(d23):,} hogares")
print()
print("Distribución de la muestra 2024 por dominio:")
t = (d24.groupby("dominio_nom")
        .agg(hogares=("pobre", "size"), pobres_muestra=("pobre", "mean"))
        .assign(pobres_muestra=lambda x: (100*x.pobres_muestra).round(1))
        .sort_values("hogares", ascending=False))
print(t.to_string())
print()
print("Un primer intento ingenuo de la tasa de pobreza:")
print(f"  hogares pobres / hogares totales = {100*d24['pobre'].mean():.2f} %")
print()
print("Pero el titular dice 27.6 %. Y ese numero es de PERSONAS, no de hogares,")
print("y esta EXPANDIDO a la poblacion. Esas dos cosas son el bloque 1.")

### La serie que hay que tener en la cabeza

Pobreza monetaria en el Perú, población afectada (INEI):

| Año | 2019 | 2020 | 2021 | 2022 | 2023 | 2024 |
|---|---|---|---|---|---|---|
| % | 20.2 | 30.1 | 25.9 | 27.5 | 29.0 | **27.6** |

El salto de 2020 es la pandemia. Lo que vamos a estudiar hoy es la última columna: si
esa caída de 29.0 a 27.6 es real o cabe dentro del ruido del muestreo.

*(Apéndice opcional al final del notebook: comparar Perú con el resto de Sudamérica
usando la API del Banco Mundial.)*

---
# Bloque 1 · Factores de expansión  ·  28 min  ·  **TEMA NUEVO**

Aquí está la pieza que faltaba en todo el curso.

## El problema

La ENAHO **no** es un muestreo aleatorio simple. Es estratificado y por conglomerados,
y sobremuestrea zonas rurales y departamentos pequeños a propósito: si sorteara al azar
puro, Madre de Dios saldría con 30 hogares y no se podría decir nada de ese departamento.

Consecuencia: **un hogar de la muestra no vale lo mismo que otro.** Un hogar rural de
Amazonas puede representar a 40 hogares del país; uno de Lima, a 1 500.

## La solución

`factor07` dice a cuántos hogares del país representa cada hogar de la muestra. Para
estimar cualquier cosa a nivel país hay que **ponderar** por ese factor.

$$\bar{x}_{ponderado} = \frac{\sum w_i x_i}{\sum w_i}$$

En numpy: `np.average(x, weights=w)`.

### Ejercicio 1.1 — ¿A cuánto expande la muestra?

Si `factor07` es «a cuántos hogares representa cada hogar», entonces su suma debería ser
**el total de hogares del Perú**. Compruébalo.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
total_hogares  = d24["w_hog"].sum()
total_personas = d24["w_per"].sum()

print(f"hogares del Peru  estimados: {total_hogares:,.0f}")
print(f"personas del Peru estimadas: {total_personas:,.0f}")
print()
print(f"tamano promedio del hogar = {total_personas/total_hogares:.2f} personas")
print()
print(f"La muestra tiene {len(d24):,} hogares y representa a {total_hogares:,.0f}.")
print(f"Es decir, cada hogar de la muestra 'vale' en promedio {total_hogares/len(d24):.0f} hogares del pais.")
print()
print("Rango de factor07:")
print(f"  minimo = {d24['factor07'].min():.1f}  (hogares muy sobremuestreados)")
print(f"  maximo = {d24['factor07'].max():.1f}  (hogares que representan a muchisimos)")
print(f"  El hogar con mas peso vale {d24['factor07'].max()/d24['factor07'].min():.0f} veces mas que el de menos peso.")

In [ ]:
# ── VERIFICACIÓN 1.1 ─────────────────────────────────────────────────────
r = [check("hogares estimados del Perú", total_hogares, 10_360_811, tol=1),
     check("personas estimadas del Perú", total_personas, 34_482_699, tol=2),
     check_bool("y el resultado es plausible: ~10.4 M de hogares y ~34.5 M de personas",
                9e6 < total_hogares < 12e6 and 32e6 < total_personas < 36e6)]
print()
print("Ese es el sentido de 'expandir': la muestra habla por todo el pais.")
print()
print("1.1 OK" if all(r) else "Revisa 1.1")

### Ejercicio 1.2 — Las tres tasas de pobreza

Ahora el punto central del bloque. Calcula la tasa de tres maneras:

1. **Sin ponderar:** hogares pobres / hogares de la muestra.
2. **Ponderada por hogares:** usando `w_hog`.
3. **Ponderada por personas:** usando `w_per`.

Solo una de las tres es la cifra oficial. Averigua cuál.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
tasa_sin_pond = 100 * d24["pobre"].mean()
tasa_pond_hog = 100 * np.average(d24["pobre"], weights=d24["w_hog"])
tasa_pond_per = 100 * np.average(d24["pobre"], weights=d24["w_per"])

print(f"1. sin ponderar           : {tasa_sin_pond:6.2f} %   <- la muestra, no el pais")
print(f"2. ponderada por hogares  : {tasa_pond_hog:6.2f} %   <- % de HOGARES pobres del pais")
print(f"3. ponderada por personas : {tasa_pond_per:6.2f} %   <- LA CIFRA OFICIAL")
print()
print("=" * 66)
print("El titular decia 27.6 %. Lo reprodujimos.")
print("=" * 66)
print()
print("Las dos correcciones que hicieron falta, y por que:")
print()
print(f"  (a) ponderar: {tasa_sin_pond:.2f} -> {tasa_pond_hog:.2f}  ({tasa_pond_hog-tasa_sin_pond:+.2f} pp)")
print("      Las zonas rurales y pobres estan SOBREmuestreadas, asi que en la muestra")
print("      hay proporcionalmente mas pobres que en el pais... o menos, segun el caso.")
print()
print(f"  (b) contar personas: {tasa_pond_hog:.2f} -> {tasa_pond_per:.2f}  ({tasa_pond_per-tasa_pond_hog:+.2f} pp)")
print("      Los hogares pobres tienen MAS miembros. Al contar personas pesan mas.")
print()
mp = np.average(d24.loc[d24.pobre==1,"mieperho"], weights=d24.loc[d24.pobre==1,"w_hog"])
mn = np.average(d24.loc[d24.pobre==0,"mieperho"], weights=d24.loc[d24.pobre==0,"w_hog"])
print(f"      miembros por hogar POBRE    : {mp:.2f}")
print(f"      miembros por hogar NO pobre : {mn:.2f}")
print(f"      -> esa diferencia de {mp-mn:.2f} personas explica los {tasa_pond_per-tasa_pond_hog:.1f} pp.")

In [ ]:
# ── VERIFICACIÓN 1.2 ─────────────────────────────────────────────────────
r = [check("tasa sin ponderar (%)", tasa_sin_pond, 20.2190, tol=1e-3),
     check("tasa ponderada por hogares (%)", tasa_pond_hog, 21.8509, tol=1e-3),
     check("tasa ponderada por personas (%) = LA OFICIAL", tasa_pond_per, 27.5795, tol=1e-3)]
print()
print("Lo que hay que llevarse del bloque:")
print("  ignorar los pesos habria dado 20.2 % en lugar de 27.6 %.")
print("  Son 7.4 puntos porcentuales, o 2.5 millones de personas.")
print()
print("Bloque 1 COMPLETO" if all(r) else "Revisa 1.2")

### Para discutir (2 min)

Un analista te entrega un reporte que dice «la pobreza es 20.2 %» y su código es
correcto: sumó bien, dividió bien, no hay bugs.

¿Qué le dirías? ¿Y qué tipo de error es este — de cálculo, de datos, o de diseño?

*(Es de diseño. Y es el mismo error de la slide del sesgo de selección de la Clase 4:
tratar una muestra no aleatoria como si lo fuera. Ningún test unitario lo detecta.)*

---
# Bloque 2 · Reconstruir la cifra desde cero  ·  20 min

Ya reprodujimos el 27.6 %, pero usando la variable `pobreza` que el INEI ya calculó.
Ahora la construimos nosotros, desde el gasto de cada hogar.

**La definición oficial, completa:**

> Un hogar es pobre si su **gasto per cápita mensual** está por debajo de la **línea de
> pobreza** de su dominio geográfico.

Dos cosas que revive esto: el gasto es una variable **continua y lognormal** (Clase 3), y
la comparación contra un umbral produce una variable Bernoulli (Clase 1).

> **Precisión importante:** la línea de pobreza **no es un cuantil**. Un cuantil se define
> por su posición dentro de una distribución (el percentil 27, por ejemplo). La línea de
> pobreza es un **umbral monetario absoluto**: el costo de una canasta básica de consumo
> que el INEI calcula para cada dominio. Que el 27.6 % de la población quede por debajo es
> el *resultado* de aplicar ese umbral, no su definición.

### Ejercicio 2.1 — Construye el gasto per cápita mensual

`gashog2d` es el gasto **anual** del **hogar**. Necesitas gasto **mensual** por
**persona**.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
d24["gpc"] = d24["gashog2d"] / d24["mieperho"] / 12
d23["gpc"] = d23["gashog2d"] / d23["mieperho"] / 12

print(d24[["gashog2d","mieperho","gpc","linea","pobre"]].head().to_string(index=False))
print()
print(f"Gasto per capita mensual (S/):")
print(f"  media   = {d24['gpc'].mean():7.2f}")
print(f"  mediana = {d24['gpc'].median():7.2f}   <- muy por debajo de la media")
print(f"  sd      = {d24['gpc'].std(ddof=1):7.2f}")
print()
print("La media supera a la mediana en un 27 %. Eso es la firma de una distribucion")
print("asimetrica con cola derecha: exactamente la lognormal de la Clase 3.")
print()
print(f"La linea de pobreza NO es un numero unico: tiene {d24['linea'].nunique()} valores distintos,")
print(f"de S/ {d24['linea'].min():.0f} a S/ {d24['linea'].max():.0f}, segun el dominio y el ano de la encuesta.")

### Ejercicio 2.2 — ¿Es lognormal?

Aplica el diagnóstico de la Clase 3: asimetría y exceso de curtosis, sobre la variable
y sobre su logaritmo.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
g = d24["gpc"]
lg = np.log(g[g > 0])

asim_gpc, curt_gpc = stats.skew(g), stats.kurtosis(g)
asim_log, curt_log = stats.skew(lg), stats.kurtosis(lg)

print(f"gpc      : asimetria = {asim_gpc:7.3f}   exceso curtosis = {curt_gpc:8.3f}")
print(f"log(gpc) : asimetria = {asim_log:7.3f}   exceso curtosis = {curt_log:8.3f}")
print()
print("Una normal daria 0 y 0. El gasto esta lejisimos; su logaritmo esta encima.")
print("Es una lognormal de manual, con datos reales del INEI.")

fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
ax[0].hist(g, bins=120, range=(0, 4000), color=MAG, alpha=0.85)
ax[0].axvline(d24["linea"].mean(), color=NAVY, lw=2.5, ls="--",
              label=f"línea media S/ {d24['linea'].mean():.0f}")
ax[0].set_title("Gasto per cápita mensual", color=NAVY, fontweight="bold")
ax[0].set_xlabel("S/ por persona al mes"); ax[0].legend(frameon=False)
ax[1].hist(lg, bins=100, color=BLUE, alpha=0.85)
ax[1].set_title("log(gasto per cápita) → normal", color=NAVY, fontweight="bold")
ax[1].set_xlabel("log(S/)")
plt.tight_layout(); plt.show()

print("En el grafico de la izquierda, todo lo que queda a la IZQUIERDA de la linea")
print("punteada es pobreza. Asi se ve la cifra oficial.")

In [ ]:
# ── VERIFICACIÓN 2.2 ─────────────────────────────────────────────────────
r = [check("asimetría del gasto", asim_gpc, 3.8019, tol=1e-3),
     check("exceso de curtosis del gasto", curt_gpc, 41.0265, tol=1e-2),
     check_bool("el logaritmo queda casi simétrico (|asimetría| < 0.15)",
                abs(asim_log) < 0.15),
     check_bool("y con curtosis casi normal (|exceso| < 0.15)", abs(curt_log) < 0.15)]
print()
print("2.2 OK" if all(r) else "Revisa 2.2")

### Ejercicio 2.3 — Construye la pobreza y compárala con la oficial

Ahora el momento del taller: define pobre como `gpc < linea` y compara tu variable con
la del INEI, hogar por hogar.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
d24["mi_pobre"] = (d24["gpc"] < d24["linea"]).astype(int)
d23["mi_pobre"] = (d23["gpc"] < d23["linea"]).astype(int)

coincidencia = 100 * (d24["mi_pobre"] == d24["pobre"]).mean()
mi_tasa = 100 * np.average(d24["mi_pobre"], weights=d24["w_per"])

print(f"Hogares donde mi definicion coincide con la del INEI: {coincidencia:.4f} %")
print(f"Discrepancias: {(d24['mi_pobre'] != d24['pobre']).sum()} de {len(d24):,} hogares")
print()
print(f"Mi tasa de pobreza (personas) : {mi_tasa:.4f} %")
print(f"La oficial                    : {100*np.average(d24['pobre'],weights=d24['w_per']):.4f} %")
print()
print("=" * 66)
print("Reconstruiste la estadistica oficial del pais con tres lineas de codigo:")
print("  gpc = gashog2d / mieperho / 12")
print("  pobre = gpc < linea")
print("  tasa  = np.average(pobre, weights=factor07 * mieperho)")
print("=" * 66)
print()
# Y la pobreza extrema, con la otra linea
d24["mi_extremo"] = (d24["gpc"] < d24["linpe"]).astype(int)
print(f"Pobreza EXTREMA (gpc < linpe): {100*np.average(d24['mi_extremo'],weights=d24['w_per']):.2f} %")
print(f"  la oficial: {100*np.average(d24['pobre_ext'],weights=d24['w_per']):.2f} %")

In [ ]:
# ── VERIFICACIÓN 2.3 ─────────────────────────────────────────────────────
r = [check("coincidencia con la variable oficial (%)", coincidencia, 100.0, tol=0.01),
     check("tu tasa de pobreza de personas (%)", mi_tasa, 27.5795, tol=1e-3)]
print()
print("Bloque 2 COMPLETO" if all(r) else "Revisa 2.3")

---
# Bloque 3 · ¿Qué tan seguro es ese 27.6 %?  ·  22 min

El 27.6 % es una **estimación** hecha con 33 691 hogares de un país de 34 millones de
personas. Toca ponerle el intervalo de la Clase 4.

Y aquí va a pasar algo: el intervalo que aprendimos **no sirve tal cual**.

### Ejercicio 3.1 — El intervalo ingenuo

Empecemos con lo que sabemos: intervalo para una proporción, con la fórmula de la
Clase 4 (`p ± 1.96 · √(p(1−p)/n)`).

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
p_hat = np.average(d24["pobre"], weights=d24["w_per"])
n = len(d24)

# El EE de una proporcion usa la MISMA p que la estimacion puntual.
# Aqui p_hat es la ponderada (27.58 %), asi que el EE ingenuo se calcula con ella.
ee_ingenuo = np.sqrt(p_hat * (1 - p_hat) / n)
ic_lo_ing = 100 * (p_hat - 1.96 * ee_ingenuo)
ic_hi_ing = 100 * (p_hat + 1.96 * ee_ingenuo)

print(f"p       = {100*p_hat:.2f} %")
print(f"n       = {n:,} hogares")
print(f"EE      = {100*ee_ingenuo:.3f} pp")
print(f"IC 95 % = [{ic_lo_ing:.2f} %, {ic_hi_ing:.2f} %]     ancho = {ic_hi_ing-ic_lo_ing:.2f} pp")
print()
print(f"Con {n:,} hogares el intervalo sale angostisimo: +/- {100*1.96*ee_ingenuo:.2f} puntos.")
print("Parece una estimacion muy precisa.")
print()
print("Pero esta formula asume MUESTREO ALEATORIO SIMPLE, y la ENAHO no lo es.")
print("Los hogares se sortean por CONGLOMERADOS: areas geograficas completas.")

### Ejercicio 3.2 — El problema de los conglomerados

Mira la estructura real de la muestra antes de seguir.

In [ ]:
# ── DEMOSTRACIÓN ─────────────────────────────────────────────────────────
g = d24.groupby("conglome")
print(f"conglomerados en 2024      : {d24['conglome'].nunique():,}")
print(f"hogares por conglomerado   : media {g.size().mean():.1f}  rango [{g.size().min()}, {g.size().max()}]")
print()
# ¿se parecen los hogares dentro de un mismo conglomerado?
tasa_cong = g["pobre"].mean()
print("Tasa de pobreza DENTRO de cada conglomerado:")
print(f"  conglomerados con 0 % de pobres  : {100*(tasa_cong==0).mean():.1f} %")
print(f"  conglomerados con 100 % de pobres: {100*(tasa_cong==1).mean():.1f} %")
print(f"  conglomerados 'mixtos'           : {100*((tasa_cong>0)&(tasa_cong<1)).mean():.1f} %")
print()
hom = 100*((tasa_cong==0).mean()+(tasa_cong==1).mean())
print(f"Casi el {hom:.0f} % de los conglomerados son HOMOGENEOS: o todos pobres o ninguno.")
print("Es una proporcion enorme, y significa que los hogares vecinos SE PARECEN.")
print("Por lo tanto cada hogar")
print("nuevo dentro del mismo conglomerado aporta MENOS informacion nueva de la que")
print("aportaria un hogar sorteado al azar en todo el pais.")
print()
print("La slide 17 de la Clase 4 lo anticipaba: 'con datos por conglomerados el EE real")
print("es mayor que s/raiz(n) y el intervalo sale falsamente angosto'.")

### Ejercicio 3.3 — El error estándar correcto, y el efecto de diseño

La fórmula que respeta el diseño (método del *conglomerado último*):

$$\widehat{Var}(\hat p) = \frac{m}{m-1} \cdot \frac{\sum_c a_c^2}{(\sum_i w_i)^2}
\qquad \text{con} \qquad a_c = \sum_{i \in c} w_i (y_i - \hat p)$$

Donde `m` es el número de conglomerados y `a_c` es la contribución de cada uno.

**La idea:** en lugar de tratar los 33 691 hogares como independientes, trata los 5 359
**conglomerados** como las unidades independientes.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
w = d24["w_per"]
total_w = w.sum()

a_c = d24.groupby("conglome").apply(lambda s: (s["w_per"] * (s["pobre"] - p_hat)).sum())
m = len(a_c)
var_diseno = (m / (m - 1)) * (a_c ** 2).sum() / total_w ** 2
ee_diseno = np.sqrt(var_diseno)

deff = (ee_diseno / ee_ingenuo) ** 2
n_efectivo = n / deff

ic_lo_d = 100 * (p_hat - 1.96 * ee_diseno)
ic_hi_d = 100 * (p_hat + 1.96 * ee_diseno)

print(f"conglomerados (m)   = {m:,}")
print()
print(f"{'':22}{'EE (pp)':>10}{'IC 95 %':>22}{'ancho':>9}")
print("-" * 64)
print(f"{'ingenuo (MAS)':22}{100*ee_ingenuo:10.3f}   [{ic_lo_ing:6.2f}, {ic_hi_ing:6.2f}] %{ic_hi_ing-ic_lo_ing:8.2f}")
print(f"{'con el diseño real':22}{100*ee_diseno:10.3f}   [{ic_lo_d:6.2f}, {ic_hi_d:6.2f}] %{ic_hi_d-ic_lo_d:8.2f}")
print("-" * 64)
print()
print(f"EFECTO DE DISENO (DEFF) = {deff:.2f}")
print(f"  el intervalo ingenuo es {np.sqrt(deff):.2f} veces mas ANGOSTO de lo que corresponde")
print()
print(f"n EFECTIVO = {n_efectivo:,.0f}")
print("=" * 66)
print(f"  Creias tener {n:,} hogares.")
print(f"  Para efectos de precision tienes {n_efectivo:,.0f}.")
print(f"  El diseno por conglomerados te 'costo' {n - n_efectivo:,.0f} hogares de precision.")
print("=" * 66)
print()
print("Por que igual conviene: sortear 33 691 hogares al azar en todo el Peru seria")
print("carisimo (un encuestador por hogar, en 1 800 distritos). Agrupar por")
print("conglomerados abarata muchisimo el trabajo de campo. El DEFF es el precio")
print("estadistico de ese ahorro logistico, y es una decision deliberada del INEI.")
print()
print("=" * 66)
print("HONESTIDAD METODOLOGICA sobre este numero")
print("=" * 66)
print("Este es un DEFF APROXIMADO, calculado con la correccion por conglomerados")
print("(metodo del 'ultimate cluster'). Dos matices que hay que declarar:")
print()
print("  1. No incorpora explicitamente la ESTRATIFICACION de la ENAHO. Ignorarla")
print("     tiende a SOBREestimar la varianza, asi que somos conservadores.")
print("     (Lo comprobamos: incluir estratos mueve el EE de 0.5253 a 0.5256 pp,")
print("      es decir, practicamente nada en este caso.)")
print()
print("  2. El INEI publica sus propios errores estandar con la metodologia oficial")
print("     completa. Nuestro numero es del orden correcto, no una replica exacta.")
print()
print("Para una estimacion de produccion se usaria un paquete de encuestas complejas")
print("(el paquete 'survey' de R, o statsmodels.survey), que maneja estratos,")
print("conglomerados y pesos de forma integrada.")

In [ ]:
# ── VERIFICACIÓN 3.3 ─────────────────────────────────────────────────────
r = [check("número de conglomerados", m, 5359, tol=0),
     check("EE ingenuo (pp)", 100*ee_ingenuo, 0.2435, tol=1e-2),
     check("EE con diseño (pp)", 100*ee_diseno, 0.5253, tol=1e-2),
     check("efecto de diseño (DEFF)", deff, 4.6551, tol=0.05),
     check("n efectivo", n_efectivo, 7238, tol=50),
     check_bool("el EE con diseño es mayor que el ingenuo", ee_diseno > ee_ingenuo)]
print()
print("Bloque 3 COMPLETO" if all(r) else "Revisa 3.3")

### Y para la media del gasto

Lo mismo aplica a cualquier estimación, no solo a proporciones. Reportar el gasto medio
sin corregir por diseño es el mismo error.

In [ ]:
# ── DEMOSTRACIÓN ─────────────────────────────────────────────────────────
gm = np.average(d24["gpc"], weights=d24["w_per"])
ee_g_ing = d24["gpc"].std(ddof=1) / np.sqrt(n)
a_g = d24.groupby("conglome").apply(lambda s: (s["w_per"] * (s["gpc"] - gm)).sum())
ee_g_dis = np.sqrt((m/(m-1)) * (a_g**2).sum() / total_w**2)

print(f"Gasto per capita medio del Peru = S/ {gm:.2f} al mes")
print(f"  IC ingenuo : [S/ {gm-1.96*ee_g_ing:.2f}, S/ {gm+1.96*ee_g_ing:.2f}]")
print(f"  IC diseño  : [S/ {gm-1.96*ee_g_dis:.2f}, S/ {gm+1.96*ee_g_dis:.2f}]")
print(f"  DEFF del gasto medio = {(ee_g_dis/ee_g_ing)**2:.2f}")
print()
print("Nota de la Clase 4: en una poblacion tan asimetrica, la MEDIANA es un mejor")
print("resumen del bienestar tipico que la media. Pero hay que ponderarla igual que todo:")
print()
print(f"  mediana MUESTRAL  : S/ {d24['gpc'].median():.2f}   <- describe la MUESTRA")
print(f"  mediana PONDERADA : S/ {mediana_ponderada(d24['gpc'], d24['w_per']):.2f}   <- describe al PAIS")
print()
print("Son unos S/ 45 de diferencia, un 7 %. La mediana muestral esta sesgada al alza")
print("porque las zonas mas pobres estan sobremuestreadas y pesan de mas en la muestra cruda.")

---
# Bloque 4 · Comparar grupos  ·  25 min

Dos preguntas que un comité haría de inmediato:

1. **¿Es distinta la pobreza entre lo urbano y lo rural?**
2. **¿Bajó la pobreza de 2023 a 2024?**

La primera es fácil. La segunda es donde se juega el taller.

### Ejercicio 4.1 — Urbano contra rural

Primero identifica el diseño (Clase 6, slide 5) y después elige la prueba.

> **Aviso metodológico, y hay que decirlo en voz alta:** las **tasas** que reportamos abajo
> están ponderadas, pero la **prueba de hipótesis** (Welch) y el **tamaño del efecto** (d de
> Cohen) se calculan sobre la muestra **sin ponderar**. Son dos cosas distintas:
>
> - las tasas estiman al **país**;
> - el Welch compara los dos grupos **de la muestra**.
>
> Hacer pruebas de hipótesis con diseño complejo requiere métodos específicos que no
> cubrimos en el curso. Para clase la aproximación sirve —la conclusión no cambia—, pero
> **no la presentes como estimación oficial de encuesta compleja**.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
u = d24[d24.area == "urbano"]
r_ = d24[d24.area == "rural"]

pob_urbano = 100 * np.average(u["pobre"], weights=u["w_per"])
pob_rural  = 100 * np.average(r_["pobre"], weights=r_["w_per"])

print("DISENO: son hogares distintos en cada grupo -> INDEPENDIENTES")
print()
print(f"{'':10}{'n':>8}{'pobreza':>10}{'gasto medio':>14}")
print("-" * 44)
print(f"{'urbano':10}{len(u):8,}{pob_urbano:9.2f} %{np.average(u['gpc'],weights=u['w_per']):13.0f}")
print(f"{'rural':10}{len(r_):8,}{pob_rural:9.2f} %{np.average(r_['gpc'],weights=r_['w_per']):13.0f}")
print("-" * 44)
print(f"brecha: {pob_rural-pob_urbano:.2f} pp de pobreza")
print()

p_levene = stats.levene(u["gpc"], r_["gpc"]).pvalue
print(f"Levene (igualdad de varianzas): p = {p_levene:.3e}  -> varianzas MUY distintas")
print("   => va Welch, no Student. (Clase 6)")
print()
res_welch = stats.ttest_ind(u["gpc"], r_["gpc"], equal_var=False)
sp = np.sqrt(((len(u)-1)*u["gpc"].var(ddof=1) + (len(r_)-1)*r_["gpc"].var(ddof=1))
             / (len(u)+len(r_)-2))
d_cohen = (u["gpc"].mean() - r_["gpc"].mean()) / sp

print("Ahora una comparacion CLASICA sobre la muestra, sin ponderar:")
print(f"  t de Welch: t = {res_welch.statistic:.2f}, gl = {res_welch.df:.0f}, p = {res_welch.pvalue:.2e}")
print(f"  d de Cohen = {d_cohen:.3f}  (efecto mediano-grande)")
print()
print("OJO con la incoherencia que esto genera si no la declaras:")
print(f"  diferencia de medias SIN ponderar : S/ {u['gpc'].mean()-r_['gpc'].mean():.0f}")
print(f"  diferencia de medias PONDERADA    : S/ {np.average(u['gpc'],weights=u['w_per'])-np.average(r_['gpc'],weights=r_['w_per']):.0f}")
print("  La segunda es la que describe al pais y es la que va en el informe.")
print("  La primera es la que corresponde al estadistico t que acabamos de calcular.")
print()
chi2, pv, dof, _ = stats.chi2_contingency(pd.crosstab(d24["area"], d24["pobre"]))
print(f"chi2 de area x pobreza: {chi2:.1f}, gl = {dof}, p = {pv:.2e}")
print("   (esta prueba la vemos en detalle en la Clase 7 -- hoy solo la usamos)")
print()
print("REPORTE, en el orden de la Clase 6 (usando las cifras PONDERADAS para el efecto):")
dif_pond = np.average(u['gpc'],weights=u['w_per']) - np.average(r_['gpc'],weights=r_['w_per'])
print(f"  el gasto per capita rural es S/ {dif_pond:.0f} menor que el urbano;")
print(f"  la pobreza rural es {pob_rural-pob_urbano:.1f} pp mayor.")
print(f"  La diferencia es estadisticamente clara (Welch sobre la muestra:")
print(f"  t = {res_welch.statistic:.1f}, gl = {res_welch.df:.0f}, p < 0.001; d = {d_cohen:.2f}).")

In [ ]:
# ── VERIFICACIÓN 4.1 ─────────────────────────────────────────────────────
r = [check("pobreza urbana (%)", pob_urbano, 24.8070, tol=1e-2),
     check("pobreza rural (%)", pob_rural, 39.3028, tol=1e-2),
     check("d de Cohen del gasto", d_cohen, 0.7512, tol=1e-3),
     check_bool("Levene rechaza igualdad de varianzas", p_levene < 0.01),
     check_bool("y por eso usaste Welch", res_welch.df < len(u)+len(r_)-2)]
print()
print("4.1 OK" if all(r) else "Revisa 4.1")

### Ejercicio 4.2 — ¿Bajó la pobreza? (el ejercicio del taller)

El titular dice que la pobreza bajó de 29.0 % a 27.6 %: **1.5 puntos porcentuales**.

La pregunta es si esa caída es real o cabe dentro del ruido del muestreo. Y aquí importa
todo lo del bloque 3: si usas el error estándar ingenuo, vas a llegar a una conclusión
distinta que si usas el correcto.

Calcula **las dos versiones** y compáralas.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
def tasa_y_ee(d):
    """Devuelve (p, ee_ingenuo, ee_diseno) de la tasa de pobreza de personas."""
    p = np.average(d["pobre"], weights=d["w_per"])
    nn = len(d)
    # coherente con el bloque 3: el EE usa la MISMA p que la estimacion puntual
    ee_i = np.sqrt(p * (1 - p) / nn)
    tot = d["w_per"].sum()
    a = d.groupby("conglome").apply(lambda s: (s["w_per"] * (s["pobre"] - p)).sum())
    mm = len(a)
    ee_d = np.sqrt((mm / (mm - 1)) * (a ** 2).sum() / tot ** 2)
    return p, ee_i, ee_d

p23, e23_ing, e23_dis = tasa_y_ee(d23)
p24, e24_ing, e24_dis = tasa_y_ee(d24)

dif = p24 - p23
ee_dif_dis = np.sqrt(e23_dis**2 + e24_dis**2)
ee_dif_ing = np.sqrt(e23_ing**2 + e24_ing**2)
z_honesto = dif / ee_dif_dis
z_ingenuo = dif / ee_dif_ing
p_honesto = 2 * stats.norm.sf(abs(z_honesto))
p_ingenuo = 2 * stats.norm.sf(abs(z_ingenuo))

print(f"2023: {100*p23:.2f} %   (EE diseño {100*e23_dis:.3f} pp)")
print(f"2024: {100*p24:.2f} %   (EE diseño {100*e24_dis:.3f} pp)")
print(f"caida = {100*dif:+.2f} pp")
print()
print("=" * 70)
print(f"{'':26}{'EE dif':>10}{'z':>9}{'p':>12}{'veredicto':>13}")
print("-" * 70)
print(f"{'ignorando el diseño':26}{100*ee_dif_ing:10.3f}{z_ingenuo:9.2f}{p_ingenuo:12.2e}   contundente")
print(f"{'con el diseño real':26}{100*ee_dif_dis:10.3f}{z_honesto:9.2f}{p_honesto:12.4f}   por los pelos")
print("=" * 70)
print()
print(f"IC 95 % de la caida (con diseño): [{100*(dif-1.96*ee_dif_dis):+.2f}, {100*(dif+1.96*ee_dif_dis):+.2f}] pp")
print()
print("MIRA EL LIMITE SUPERIOR: -0.03 pp. Esta a TRES CENTESIMAS de punto de cero.")
print()
print("La lectura honesta es:")
print("  'La pobreza bajo 1.5 puntos porcentuales. La caida alcanza significancia")
print("   estadistica por muy poco margen (p = 0.046), y el intervalo de confianza")
print("   admite caidas de hasta 2.9 pp o de practicamente cero. Hay evidencia de")
print("   mejora, pero no es concluyente con un solo ano de comparacion.'")
print()
print(f"Si hubieras ignorado el diseno, el z habria salido {z_ingenuo/z_honesto:.1f} veces mas grande")
print("y habrias reportado una caida contundente que los datos no respaldan.")

In [ ]:
# ── VERIFICACIÓN 4.2 ─────────────────────────────────────────────────────
r = [check("pobreza 2023 (%)", 100*p23, 29.0459, tol=1e-2),
     check("pobreza 2024 (%)", 100*p24, 27.5795, tol=1e-2),
     check("caída (pp)", 100*dif, -1.4665, tol=1e-2),
     check("z con diseño", z_honesto, -1.9974, tol=1e-2),
     check_bool("con diseño la caída es significativa por poco (0.01 < p < 0.05)",
                0.01 < 2*stats.norm.sf(abs(z_honesto)) < 0.05),
     check_bool("y sin diseño parecería mucho más contundente",
                abs(z_ingenuo) > 2*abs(z_honesto))]
print()
print("4.2 OK" if all(r) else "Revisa 4.2")

### Un matiz de diseño que no conviene esconder

Traté las muestras de 2023 y 2024 como **independientes**, que es lo que hace el INEI en
sus publicaciones. Pero la ENAHO tiene un **panel rotativo**: parte de los hogares se
vuelve a visitar al año siguiente.

Compruébalo.

In [ ]:
# ── DEMOSTRACIÓN ─────────────────────────────────────────────────────────
for d in (d23, d24):
    d["id"] = (d["conglome"].astype(str) + "-" + d["vivienda"].astype(str)
               + "-" + d["hogar"].astype(str))
comunes = set(d23["id"]) & set(d24["id"])
print(f"identificadores presentes en los dos años: {len(comunes):,}")
print(f"  = {100*len(comunes)/len(d24):.1f} % de la muestra de 2024")
print()

# PASO 1: ¿son REALMENTE los mismos hogares, o coincidencias del marco muestral?
mg = d23.merge(d24, on="id", suffixes=("_23", "_24"))
print("¿Son de verdad los mismos hogares?")
print(f"  mismo dominio : {100*(mg.dominio_23==mg.dominio_24).mean():.1f} %")
print(f"  mismo estrato : {100*(mg.estrato_23==mg.estrato_24).mean():.1f} %")
print(f"  correlacion del numero de miembros entre anios: {mg['mieperho_23'].corr(mg['mieperho_24']):.3f}")
print("  -> Si fueran hogares distintos que coinciden por azar del marco, el dominio")
print("     tambien coincidiria, pero mieperho NO estaria correlacionado. Lo esta.")
print("     Confirmado: es panel rotativo real.")
print()

# PASO 2: medir la correlacion de la condicion de pobreza entre anios
r_panel = np.corrcoef(mg["pobre_23"], mg["pobre_24"])[0, 1]
frac = len(mg) / len(d24)
print(f"Correlacion de 'pobre' en los hogares del panel: r = {r_panel:.3f}")
print(f"Fraccion solapada: f = {frac:.3f}")
print()

# PASO 3: recalcular el EE de la diferencia INCORPORANDO la covarianza
# APROXIMACION: Cov ~ f * r * EE23 * EE24, donde f es la fraccion solapada.
# Una estimacion rigurosa requeriria replicacion (bootstrap o jackknife) sobre el
# diseno completo, con estratos y conglomerados. Esto da el ORDEN DE MAGNITUD y,
# sobre todo, el SIGNO -- que es lo que necesitamos para saber si somos conservadores.
cov_aprox = frac * r_panel * e23_dis * e24_dis
ee_corr = np.sqrt(e23_dis**2 + e24_dis**2 - 2*cov_aprox)
z_corr = dif / ee_corr
p_corr = 2 * stats.norm.sf(abs(z_corr))

print(f"{'':32}{'EE dif':>10}{'z':>9}{'p':>10}")
print("-" * 62)
print(f"{'asumiendo independencia':32}{100*ee_dif_dis:10.4f}{z_honesto:9.3f}{2*stats.norm.sf(abs(z_honesto)):10.4f}")
print(f"{'con la covarianza medida':32}{100*ee_corr:10.4f}{z_corr:9.3f}{p_corr:10.4f}")
print("-" * 62)
print()
if cov_aprox > 0:
    print("La covarianza resulto POSITIVA. Eso significa que asumir independencia")
    print("SOBREestima el error estandar, es decir, somos conservadores. La conclusion")
    print("del ejercicio anterior se sostiene... pero ahora esta COMPROBADA, no supuesta.")
else:
    print("La covarianza NO resulto positiva: el argumento de que somos conservadores")
    print("NO se sostiene con estos datos.")
print()
print("Y fijate en lo que importa de verdad: el p se movio de 0.046 a 0.034 por")
print("cambiar UN supuesto. Sigue siendo fronterizo. Eso es lo que significa que un")
print("resultado sea 'sensible al metodo', y es la razon para no titular")
print("'demostramos que la pobreza bajo'.")
print()
print("ALCANCE DE ESTE CALCULO, para no sobrevenderlo:")
print("  la covarianza se estimo de forma aproximada (fraccion solapada x correlacion).")
print("  Lo que este ejercicio establece con solidez es el SIGNO -- positivo -- y por")
print("  tanto la direccion del sesgo. El valor exacto del p corregido requeriria")
print("  metodos de replicacion (bootstrap o jackknife) sobre el diseno completo.")
print("  Para la conclusion del taller basta con el signo: somos conservadores.")
print()
print("Leccion de la Clase 6, slide 5: identifica el diseno antes de elegir la prueba.")
print("Y a veces el mundo real no cae limpio en ninguna de las tres casillas.")

In [ ]:
# ── VERIFICACIÓN 4.2b ────────────────────────────────────────────────────
r = [check_bool("confirmaste que el solapamiento es panel real (correlación de mieperho > 0.5)",
                mg["mieperho_23"].corr(mg["mieperho_24"]) > 0.5),
     check_bool("la correlación de pobreza entre años es positiva", r_panel > 0),
     check_bool("y por tanto asumir independencia es conservador", cov_aprox > 0),
     check_bool("pero el resultado sigue siendo fronterizo (0.01 < p < 0.05)",
                0.01 < p_corr < 0.05,
                "si sale fuera de ese rango, la conclusión del bloque cambia")]
print()
print("Lo que este ejercicio enseña, y vale mas que el numero:")
print("  afirmar 'esto es conservador' SIN medirlo es exactamente el tipo de")
print("  razonamiento plausible-pero-no-verificado que este curso combate.")
print("  Medirlo cuesta cinco lineas.")
print()
print("4.2b OK" if all(r) else "Revisa 4.2b")

### Ejercicio 4.3 — Comparar los ocho dominios

Última pregunta: ¿hay diferencias entre los ocho dominios geográficos? Aquí vuelve el
problema de las comparaciones múltiples (Clase 5).

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
tabla_dom = (d24.groupby("dominio_nom")
             .apply(lambda s: pd.Series({
                 "n": len(s),
                 "pobreza_%": 100*np.average(s["pobre"], weights=s["w_per"]),
                 "gasto_medio": np.average(s["gpc"], weights=s["w_per"]),
             }))
             .sort_values("pobreza_%", ascending=False))
print(tabla_dom.round(2).to_string())
print()
k = d24["dominio_nom"].nunique()
n_comparaciones = k * (k - 1) // 2
alpha_bonf = 0.05 / n_comparaciones
print(f"Con {k} dominios hay {n_comparaciones} comparaciones por pares.")
print(f"P(al menos un falso positivo) sin corregir = {100*(1-0.95**n_comparaciones):.1f} %")
print(f"alpha de Bonferroni = {alpha_bonf:.5f}")
print()
print("Es decir: si comparas los 8 dominios entre si sin corregir, la probabilidad de")
print("encontrar al menos una diferencia falsa es del 76 %. Casi seguro.")
print()
print("Y ademas cada tasa tiene su propio EE de diseno, que aqui ni calculamos.")
print("Con 8 dominios, la comparacion honesta requiere corregir Y respetar el diseno.")
print()
print(f"Lo que SI se puede afirmar sin ambiguedad: la brecha entre el dominio mas pobre")
print(f"({tabla_dom.index[0]}, {tabla_dom['pobreza_%'].iloc[0]:.1f} %) y el menos pobre")
print(f"({tabla_dom.index[-1]}, {tabla_dom['pobreza_%'].iloc[-1]:.1f} %) es de")
print(f"{tabla_dom['pobreza_%'].iloc[0]-tabla_dom['pobreza_%'].iloc[-1]:.1f} puntos porcentuales.")

In [ ]:
# ── VERIFICACIÓN 4.3 ─────────────────────────────────────────────────────
r = [check("número de comparaciones por pares", n_comparaciones, 28, tol=0),
     check("alpha de Bonferroni", alpha_bonf, 0.05/28, tol=1e-6),
     check_bool("identificaste Sierra Norte como el dominio más pobre",
                tabla_dom.index[0] == "Sierra Norte"),
     check_bool("y Costa Centro como el menos pobre",
                tabla_dom.index[-1] == "Costa Centro")]
print()
print("Bloque 4 COMPLETO" if all(r) else "Revisa 4.3")

---
# Bloque 5 · El informe  ·  13 min

Todo lo anterior no sirve si el informe no lo comunica. Este bloque **no tiene respuesta
numérica única**: se te pide escribir.

Recuerda la slide de redacción de la Clase 6: diagnóstico de supuestos, prueba con sus
grados de libertad, efecto en unidades reales, tamaño del efecto, cuántas comparaciones,
y **qué no permite concluir la muestra**.

In [ ]:
# ── SOLUCIÓN PROPUESTA ───────────────────────────────────────────────────
informe = f"""
ESTIMACION DE LA POBREZA MONETARIA · PERU 2024
Fuente: INEI, ENAHO 2024, modulo 34. n = {len(d24):,} hogares.

1. RESULTADO PRINCIPAL
La pobreza monetaria afecto al {100*p_hat:.1f} % de la poblacion en 2024
(IC 95 %: {100*(p_hat-1.96*ee_diseno):.1f} % a {100*(p_hat+1.96*ee_diseno):.1f} %), equivalente a unos
{p_hat*d24['w_per'].sum()/1e6:.1f} millones de personas. La estimacion se obtuvo comparando el
gasto per capita mensual de cada hogar contra la linea de pobreza de su
dominio, ponderando por el factor de expansion multiplicado por el numero
de miembros del hogar.

El intervalo reportado corrige por conglomeracion mediante el metodo del
ultimate cluster. El efecto de diseno aproximado es DEFF = {deff:.1f}, equivalente
a un tamano de muestra efectivo de {n_efectivo:,.0f} hogares. Un intervalo calculado
bajo el supuesto de muestreo aleatorio simple habria sido {np.sqrt(deff):.1f} veces mas
angosto y habria exagerado la precision. Este DEFF es una aproximacion
didactica: no incorpora explicitamente la estratificacion (lo que lo hace
conservador) y no reemplaza los errores estandar oficiales del INEI.

2. EVOLUCION RESPECTO A 2023
La tasa paso de {100*p23:.1f} % a {100*p24:.1f} %, una caida de {abs(100*dif):.1f} puntos
porcentuales (IC 95 %: {abs(100*(dif+1.96*ee_dif_dis)):.1f} a {abs(100*(dif-1.96*ee_dif_dis)):.1f} pp de reduccion;
z = {z_honesto:.2f}, p = {2*stats.norm.sf(abs(z_honesto)):.3f}). Es un resultado FRONTERIZO y
sensible al metodo: al incorporar la covarianza inducida por el panel
rotativo, el p-valor pasa a {p_corr:.3f}. La conclusion prudente es que hay
evidencia de mejora, no que la mejora este establecida. No corresponde
afirmar que "se demostro" una reduccion.

3. BRECHA URBANO-RURAL
La pobreza rural ({pob_rural:.1f} %) supera a la urbana ({pob_urbano:.1f} %) en
{pob_rural-pob_urbano:.1f} puntos. El gasto per capita medio, en estimaciones
ponderadas, difiere en S/ {dif_pond:.0f} mensuales (urbano S/ {np.average(u['gpc'],weights=u['w_per']):.0f},
rural S/ {np.average(r_['gpc'],weights=r_['w_per']):.0f}).

La diferencia es estadisticamente clara. El contraste formal se realizo
sobre la muestra sin ponderar (t de Welch = {res_welch.statistic:.1f}, gl = {res_welch.df:.0f},
p < 0.001; d de Cohen = {d_cohen:.2f}), lo que constituye una aproximacion: las
pruebas de hipotesis bajo diseno complejo requieren metodos especificos.
Se uso Welch y no Student porque las varianzas de los dos grupos difieren
(Levene p < 0.001).

4. LIMITACIONES
- La comparacion 2023-2024 se trato como muestras independientes, pero la
  ENAHO tiene panel rotativo: el {100*frac:.0f} % de los hogares se repite entre anos.
  Se verifico empiricamente que la correlacion de la condicion de pobreza
  entre anios es positiva (r = {r_panel:.2f}), por lo que asumir independencia
  sobreestima el error estandar y la prueba resulta conservadora.
- Las pruebas de hipotesis entre grupos (Welch, chi2) se calcularon sobre la
  muestra sin ponderar; las tasas y medias reportadas si estan ponderadas.
  Para inferencia con diseno complejo se requieren metodos especificos.
- Las comparaciones entre los 8 dominios geograficos ({n_comparaciones} pares posibles)
  no se reportan como pruebas individuales: sin correccion por
  comparaciones multiples la probabilidad de al menos un falso positivo
  seria del {100*(1-0.95**n_comparaciones):.0f} %.
- La pobreza monetaria mide capacidad de gasto en un momento del ano. No
  captura pobreza estructural; para eso se usan las NBI, disponibles en el
  mismo dataset.
- Un solo ano de comparacion no permite afirmar una tendencia. Se
  recomienda evaluar la serie 2019-2024 completa.
"""
print(informe)
print()
print("=" * 70)
print("Lo que hace auditable a este informe:")
print("  - reporta el intervalo, no solo el punto")
print("  - justifica el metodo (DEFF, Levene) en lugar de asumirlo")
print("  - da el efecto en soles y en puntos, no solo el p-valor")
print("  - dice explicitamente que NO se puede concluir")
print("=" * 70)

In [ ]:
# ── VERIFICACIÓN 5 ──────────────────────────────────────────────────────
r = [check_bool("escribiste un informe sustantivo", len(informe.strip()) > 300),
     check_bool("mencionas un intervalo de confianza",
                any(k in informe.upper() for k in ["IC ", "INTERVALO", "IC:"])),
     check_bool("mencionas el efecto de diseño o el n efectivo",
                any(k in informe.upper() for k in ["DEFF", "DISENO", "DISEÑO", "EFECTIVO"])),
     check_bool("incluyes al menos una limitación",
                any(k in informe.upper() for k in ["LIMITACI", "NO PERMITE", "PRUDENTE", "NO CAPTURA"]))]
print()
print("TALLER COMPLETO" if all(r) else "Completa el informe del bloque 5")

---
# Cierre

### Checklist de salida

- [ ] Sé qué es un factor de expansión y por qué sin él la cifra está mal.
- [ ] Sé que la pobreza oficial se pondera por **personas**, no por hogares.
- [ ] Reconstruí la cifra oficial del país desde el gasto de cada hogar.
- [ ] Sé calcular un efecto de diseño y explicar qué es el n efectivo.
- [ ] Sé que un IC ingenuo sobre datos por conglomerados es falsamente angosto.
- [ ] Identifiqué el diseño antes de elegir la prueba, y noté que era mixto.
- [ ] Escribí un informe que dice también lo que no se puede concluir.

### Los cinco números del taller

| | |
|---|---|
| Cifra oficial reproducida | **27.58 %** (INEI publica 27.6 %) |
| Coincidencia de mi variable con la oficial | **100.00 %** |
| Sin ponderar habría dicho | 20.22 % → error de **7.4 pp** |
| Efecto de diseño | **DEFF ≈ 4.66** → n efectivo **7 238** de 33 691 |
| ¿Bajó la pobreza? | **fronterizo**: p = 0.046 con diseño, 0.034 con la covarianza, 0.000023 sin diseño |

### Lo que este taller demostró

Las seis clases anteriores no fueron ejercicios de pizarra. Cada una apareció aquí, con
datos del INEI, y en cada punto la decisión metodológica **cambió la respuesta**:

- ignorar los pesos habría movido la cifra 7.4 puntos;
- ignorar el diseño habría hecho el intervalo 2.2 veces más angosto;
- y habría convertido una caída dudosa en un titular contundente.

La estadística aplicada no es elegir la función correcta de scipy. Es saber qué le pasó
a los datos antes de llegar a tus manos.

### Apéndice opcional · Perú en Sudamérica

Si quieres el contexto regional, la API del Banco Mundial da la pobreza a paridad de
poder de compra para los 12 países. No lo ejecutamos en clase por depender de red:

```python
!pip install wbgapi -q
import wbgapi as wb
paises = ['ARG','BOL','BRA','CHL','COL','ECU','GUY','PRY','PER','SUR','URY','VEN']
# SI.POV.UMIC = pobreza a $6.85/día PPP 2017
wb.data.DataFrame('SI.POV.UMIC', paises, mrnev=1)
```

Advertencia: esas cifras usan una línea internacional distinta de la peruana, así que
**no son comparables** con el 27.6 % de hoy. Sirven para ordenar países entre sí, no para
sustituir la medición nacional.

---
*Estadística Descriptiva e Inferencial · Taller integrador · ENAHO 2023–2024 · INEI*